# Component 2 - Model Experiments

Compares models for predicting each prop stat (points, rebounds, assists, threes), using `scripts/model_experiments.py` on the feature files (run *Player Stats Ingestion* and *Feature Engineering* first). Takes about 3 minutes.

**Split (by time, never random):** train on last season before February, tune on the rest of last season, test on this season so far after refitting on both.

**Scores:**
- **MAE**: average miss of the predicted number.
- **Log score**: how likely the actual result was under the predicted distribution (each prediction becomes a negative binomial). Lower is better; it rewards getting the spread right, not just the middle.
- **Brier (over proxy line)**: accuracy of P(over) at a line near the player's 10-game average, about where books open, since historical lines aren't stored.
- **Vegas comparison**: where the odds snapshot overlaps the test games, one row per player prop with the books' prices and no-vig probability next to each model's, and the result. One night of games, so treat the totals as a sanity check.

In [ ]:
LEAGUE = "nba"

In [ ]:
import sys
import warnings
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "model_experiments.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats, build_features, model_experiments as me
for m in (fetch_player_stats, build_features, me):
    importlib.reload(m)  # pick up script edits without restarting the kernel

import pandas as pd
warnings.filterwarnings("ignore")
ORDER = ["Last-10 average", "Last-20 average", "Ridge regression", "Poisson regression", "Random forest",
         "Gradient boosting", "Gradient boosting (Poisson)", "Ensemble (RF + GB)"]
TARGETS = ["pts", "reb", "ast", "fg3m"]

## Run

In [ ]:
results, props, parts, preds = me.run(LEAGUE)

## Test season results

In [ ]:
def table(metric, split="test"):
    return results[results["split"] == split].pivot(index="model", columns="target", values=metric).loc[ORDER, TARGETS].round(4)

for metric in ("MAE", "Log score", "Brier (over proxy line)"):
    print(metric)
    display(table(metric))

## Validation results (last season, Feb onward)

In [ ]:
for metric in ("MAE", "Log score", "Brier (over proxy line)"):
    print(metric)
    display(table(metric, "valid"))

## Vegas comparison, prop by prop

One row per player prop in the odds snapshot that overlaps the test games:

- **books**: books quoting that exact line; **main_line**: the line most books quote for that player and market.
- **best_over / best_under**: best American odds on each side, and the book offering it.
- **vegas_p_over**: average no-vig probability of the Over across those books (Novig placeholder prices dropped).
- **model_mean / model_p_over**: the ensemble's predicted stat and its probability of the Over.
- **edge**: model minus Vegas. Positive leans Over, negative leans Under.
- **actual / result**: what happened.
- **p_over: ...**: every other model's probability, for comparison.

In [ ]:
props.sort_values("edge", key=abs, ascending=False).head(25)

Look up one player:

In [ ]:
PLAYER = "Josh Giddey"
props[props["player"] == PLAYER]

### Totals from the per-prop rows

Brier score per market (lower is better), for all props and for main lines quoted by 3+ books.

In [ ]:
order = ["Vegas consensus (no vig)"] + ORDER
for title, kw in (("All props", {}), ("Main lines, 3+ books", {"main_only": True, "min_books": 3})):
    s = me.vegas_summary(props, **kw)
    print(f"{title}: {s.groupby('market')['props'].first().to_dict()}")
    display(s.pivot(index="model", columns="market", values="Brier").loc[order].round(4))

## Calibration

When the model says 60% over, does it go over about 60% of the time? Test season, proxy lines, all stats pooled.

In [ ]:
me.calibration(parts, preds, "Ensemble (RF + GB)").round(3)

## Chosen settings and dispersion

In [ ]:
results[results["split"] == "test"].pivot(index="model", columns="target", values="settings").loc[ORDER[2:7], TARGETS]

In [ ]:
results.to_csv(me.OUT_DIR / f"{LEAGUE}_model_results.csv", index=False)
if props is not None:
    props.to_csv(me.OUT_DIR / f"{LEAGUE}_vegas_props.csv", index=False)